# 7.1 端云协同推理

在端侧设备计算能力不足时，通过端云协同策略实现大模型能力的按需获取，同时保护用户隐私。端云协同是产业界大规模部署LLM的核心架构模式。

## 什么是端云协同推理？

端云协同推理是将推理任务在端侧和云端之间合理分配的策略。简单任务在端侧完成（低延迟、保护隐私），复杂任务卸载到云端（高精度、大模型能力）。

## 为什么用端云协同推理？

1. **端侧能力有限**：端侧设备无法运行超大模型（如GPT-4），但小模型能力不足
2. **隐私保护**：敏感数据（医疗、金融）不应上传云端
3. **延迟优化**：简单请求端侧处理更快，复杂请求才需云端
4. **成本控制**：减少云端API调用次数，降低服务成本
5. **离线可用**：网络不可用时端侧仍可提供基础服务

## 端云协同推理的数学原理

**模型拆分推理**：将$L$层模型分为端侧$L_e$层和云端$L_c$层：
$$h_e = f_{1:L_e}(x), \quad y = f_{L_e+1:L}(h_e)$$
延迟模型：$T_{\text{split}} = T_{\text{edge}} + T_{\text{transfer}} + T_{\text{cloud}}$

其中传输延迟：$T_{\text{transfer}} = \frac{|h_e| \cdot d_{\text{elem}}}{\text{BW}} + T_{\text{latency}}$
- $|h_e| = L_{\text{seq}} \times d_{\text{hidden}}$：中间特征元素数
- $d_{\text{elem}}$：每个元素的字节数（FP32=4, FP16=2, INT8=1）
- $\text{BW}$：网络带宽（WiFi6~1.2Gbps, 5G~100Mbps, 4G~20Mbps）
- $T_{\text{latency}}$：网络往返延迟（WiFi~5ms, 5G~15ms, 4G~50ms）

**自适应路由**：根据输入复杂度决定端侧还是云端处理：
$$\text{route}(x) = \begin{cases} \text{edge} & \text{if } \text{conf}(f_{\text{edge}}(x)) > \tau \\ \text{cloud} & \text{otherwise} \end{cases}$$

**端侧处理比例**：$\rho = P(\text{conf} > \tau)$，越大则云端负载越低、成本越低

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import gc

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

### 模型拆分推理（Split Computing）

#### 什么是模型拆分推理？

将$L$层模型分为端侧$L_e$层和云端$L_c$层，端侧执行前半部分，中间特征上传云端，云端执行后半部分并返回结果。

#### 为什么用模型拆分推理？

1. **端侧算力不足**：端侧设备无法运行完整大模型，但可以运行前几层
2. **隐私保护**：原始数据不上传云端，仅上传中间特征（不可逆）
3. **带宽节省**：中间特征可量化压缩后传输

#### 模型拆分的数学原理

拆分点选择：
$$l^* = \arg\min_l (T_{\text{edge}}(l) + T_{\text{transfer}}(l) + T_{\text{cloud}}(l))$$

其中：
- $T_{\text{edge}}(l)$：端侧执行前$l$层的延迟
- $T_{\text{transfer}}(l)$：传输中间特征的延迟$= \frac{|h_l| \cdot d}{\text{BW}} + T_{\text{latency}}$
- $T_{\text{cloud}}(l)$：云端执行后$L-l$层的延迟
- $|h_l| \cdot d$：第$l$层输出的数据量（序列长度$\times$隐藏维度$\times$元素字节数）

#### 中间特征压缩

中间特征$h_l \in \mathbb{R}^{L_{\text{seq}} \times d_{\text{hidden}}}$的传输量：
- FP32：$L_{\text{seq}} \times d_{\text{hidden}} \times 4$ bytes
- FP16量化：传输量减半，精度损失可忽略
- INT8量化：传输量降至1/4，需校准量化参数
- 特征蒸馏：训练小型投影头将$d_{\text{hidden}}$压缩到$d_{\text{compact}}$

#### 隐私增强

中间特征并非完全不可逆，攻击者可能通过反演攻击重建输入。增强隐私的方法：
- **噪声注入**：$h'_l = h_l + \epsilon, \epsilon \sim \mathcal{N}(0, \sigma^2 I)$，牺牲少量精度
- **特征扰动**：随机丢弃部分特征维度
- **加密传输**：TLS 1.3加密中间特征，防止中间人攻击

In [ ]:
class SplitTransformer(nn.Module):
    """可拆分的Transformer"""
    def __init__(self, dim=64, n_layers=6, n_heads=4, split_point=3):
        super().__init__()
        self.split_point = split_point
        self.edge_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=dim, nhead=n_heads, batch_first=True)
            for _ in range(split_point)
        ])
        self.cloud_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=dim, nhead=n_heads, batch_first=True)
            for _ in range(split_point, n_layers)
        ])

    def edge_forward(self, x):
        for layer in self.edge_layers:
            x = layer(x)
        return x

    def cloud_forward(self, x):
        for layer in self.cloud_layers:
            x = layer(x)
        return x

    def forward(self, x):
        x = self.edge_forward(x)
        x = self.cloud_forward(x)
        return x

dim, n_layers, n_heads = 64, 6, 4
model = SplitTransformer(dim, n_layers, n_heads, split_point=3)

edge_params = sum(p.numel() for p in model.edge_layers.parameters())
cloud_params = sum(p.numel() for p in model.cloud_layers.parameters())
total_params = edge_params + cloud_params

x = torch.randn(2, 16, dim)
with torch.no_grad():
    intermediate = model.edge_forward(x)
    final = model.cloud_forward(intermediate)
    full = model(x)

input_bytes = x.numel() * 4
intermediate_bytes = intermediate.numel() * 4
output_bytes = final.numel() * 4

print(f"=== 模型拆分推理 ===")
print(f"总层数: {n_layers}, 拆分点: 第{3}层")
print(f"端侧参数: {edge_params:,} ({edge_params/total_params:.1%})")
print(f"云端参数: {cloud_params:,} ({cloud_params/total_params:.1%})")
print(f"\n通信量分析:")
print(f"  原始输入: {input_bytes/1024:.2f} KB")
print(f"  中间特征(FP32): {intermediate_bytes/1024:.2f} KB")
print(f"  中间特征(FP16): {intermediate_bytes/2/1024:.2f} KB")
print(f"  中间特征(INT8): {intermediate_bytes/4/1024:.2f} KB")
print(f"  输出: {output_bytes/1024:.2f} KB")

print(f"\n--- 网络传输延迟估算 ---")
networks = [
    ('WiFi 6', 1.2e9, 5e-3),
    ('5G', 100e6, 15e-3),
    ('4G LTE', 20e6, 50e-3),
]
for net_name, bw, latency in networks:
    transfer_time = intermediate_bytes * 8 / bw + latency
    transfer_time_int8 = intermediate_bytes / 4 * 8 / bw + latency
    print(f"  {net_name}: FP32={transfer_time*1000:.1f}ms, INT8={transfer_time_int8*1000:.1f}ms")

print(f"\n隐私保护: 上传中间特征而非原始输入，云端无法直接还原原始数据")
print(f"增强隐私: 可对中间特征添加噪声或使用TLS加密传输")

### 自适应推理路由

#### 什么是自适应推理路由？

根据输入复杂度动态决定请求在端侧还是云端处理。简单请求端侧处理（低延迟），复杂请求卸载到云端（高精度）。

#### 为什么需要自适应路由？

1. **请求复杂度不均**：简单请求（闲聊）端侧即可处理，复杂请求（推理、创作）需要云端大模型
2. **成本优化**：减少不必要的云端API调用
3. **延迟优化**：简单请求避免网络往返延迟

#### 自适应路由的数学原理

路由决策：
$$	ext{route}(x) = \begin{cases} \text{edge} & \text{if } \text{conf}(f_{\text{edge}}(x)) > \tau \\ \text{cloud} & \text{otherwise} \end{cases}$$

其中：
- $\text{conf}(f_{\text{edge}}(x))$：端侧模型的预测置信度
- $\tau$：置信度阈值，控制端侧处理比例
- 端侧处理比例$\rho = P(\text{conf} > \tau)$，越大则云端负载越低

#### 产业级路由策略

1. **置信度路由**：端侧模型输出softmax最大概率作为置信度
2. **复杂度估计路由**：轻量级分类器预测输入复杂度（token数、语义复杂度）
3. **多阈值路由**：不同任务类型使用不同阈值（翻译$\tau_1$、摘要$\tau_2$、推理$\tau_3$）
4. **级联路由**：端侧先尝试，低置信度时再请求云端（带缓存）
5. **A/B测试**：线上实验确定最优阈值，平衡端侧处理率和准确率

In [ ]:
class AdaptiveInferenceRouter:
    """自适应推理路由器
    注意：本实现为教学演示，复杂度估计器为随机初始化的线性层，
    路由决策不反映真实复杂度。实际应用需训练复杂度预测模型或使用启发式规则。"""
    def __init__(self, edge_model, cloud_model, complexity_threshold=0.5):
        self.edge_model = edge_model
        self.cloud_model = cloud_model
        self.threshold = complexity_threshold
        self.complexity_estimator = nn.Linear(64, 1)

    def estimate_complexity(self, x):
        """估算输入复杂度"""
        with torch.no_grad():
            feat = x.mean(dim=1)
            complexity = torch.sigmoid(self.complexity_estimator(feat))
        return complexity.squeeze()

    def route(self, x):
        """根据复杂度路由到端侧或云端"""
        complexity = self.estimate_complexity(x)
        results = []
        for i in range(x.shape[0]):
            if complexity[i].item() < self.threshold:
                with torch.no_grad():
                    out = self.edge_model(x[i:i+1])
                results.append(('edge', out))
            else:
                with torch.no_grad():
                    out = self.cloud_model(x[i:i+1])
                results.append(('cloud', out))
        return results

edge_model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 64))
cloud_model = nn.Sequential(
    nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 64),
    nn.ReLU(), nn.Linear(64, 64)
)

router = AdaptiveInferenceRouter(edge_model, cloud_model, complexity_threshold=0.5)
x = torch.randn(8, 4, 64)
results = router.route(x)

edge_count = sum(1 for r, _ in results if r == 'edge')
cloud_count = sum(1 for r, _ in results if r == 'cloud')

print(f"=== 自适应推理路由 ===")
print(f"总请求数: {len(results)}")
print(f"端侧处理: {edge_count} ({edge_count/len(results):.0%})")
print(f"云端处理: {cloud_count} ({cloud_count/len(results):.0%})")

print(f"\n--- 不同阈值对端侧处理率的影响 ---")
for tau in [0.3, 0.5, 0.7]:
    router_t = AdaptiveInferenceRouter(edge_model, cloud_model, complexity_threshold=tau)
    res = router_t.route(x)
    ec = sum(1 for r, _ in res if r == 'edge')
    print(f"  τ={tau}: 端侧处理率={ec/len(res):.0%}")

print(f"\n--- 产业级路由成本分析 ---")
edge_cost_per_req = 0.0
cloud_cost_per_req = 0.002
edge_latency = 0.05
cloud_latency = 0.3
for rho in [0.3, 0.5, 0.7, 0.9]:
    avg_cost = rho * edge_cost_per_req + (1 - rho) * cloud_cost_per_req
    avg_latency = rho * edge_latency + (1 - rho) * cloud_latency
    print(f"  端侧率={rho:.0%}: 平均成本=${avg_cost:.4f}/req, 平均延迟={avg_latency*1000:.0f}ms")

print(f"\n优势: 简单请求端侧处理(低延迟)，复杂请求云端处理(高精度)")
print(f"关键: 阈值τ的选择需在线上A/B测试中确定")

### 端云推测验证（Edge-Cloud Speculative Decoding）

#### 什么是端云推测验证？

端侧小模型快速生成候选token，云端大模型并行验证，接受正确的token、拒绝错误的token。结合了投机解码和端云协同的优势。

#### 为什么端云推测验证有效？

1. **利用网络并行性**：云端验证$k$个token只需1次前向传播，网络延迟被分摊
2. **高接受率**：端侧小模型与云端大模型分布接近时，大部分候选被接受
3. **输出分布不变**：通过拒绝采样保证输出与云端大模型完全一致

#### 端云推测验证的数学原理

接受概率（拒绝采样）：
$$P(\text{accept } x_t) = \min\left(1, \frac{p_{\text{cloud}}(x_t)}{p_{\text{edge}}(x_t)}\right)$$

拒绝时的修正采样：
$$x_t \sim \text{normalize}(\max(0, p_{\text{cloud}}(x) - p_{\text{edge}}(x)))$$

理论加速比：
$$\text{Speedup} = \frac{1}{1 - \alpha \cdot P(\text{accept})}$$

其中$\alpha$为每次推测的token数，$P(\text{accept})$为平均接受率。

#### 关键约束

- 端侧draft模型必须是云端target模型的近似，否则接受率低，反而更慢
- $k$的选择需平衡：$k$越大单次验证的期望接受数越多，但拒绝时浪费的计算也越多
- 典型配置：$k=5$, 接受率$\approx 80\%$时，加速比约$2.5\times$

In [ ]:
class EdgeCloudSpeculativeDecoding:
    """端云推测解码: 端侧draft + 云端verify
    注意：本演示使用简化的MLP模型模拟推测解码流程，实际应用中应使用语言模型。"""
    def __init__(self, edge_model, cloud_model, k=5):
        self.edge_model = edge_model
        self.cloud_model = cloud_model
        self.k = k

    def decode_step(self, x):
        """一步推测解码"""
        draft_tokens = []
        with torch.no_grad():
            for _ in range(self.k):
                logits = self.edge_model(x)
                token = logits.argmax(dim=-1)[:, -1:]
                draft_tokens.append(token)
                x = torch.cat([x, token.float()], dim=1)

        with torch.no_grad():
            cloud_logits = self.cloud_model(x)
            cloud_tokens = cloud_logits.argmax(dim=-1)

        n_accepted = 0
        for i, dt in enumerate(draft_tokens):
            ct = cloud_tokens[:, x.shape[1] - self.k + i]
            if dt.squeeze().item() == ct.item():
                n_accepted += 1
            else:
                break

        return n_accepted, len(draft_tokens)

edge = nn.Sequential(nn.Linear(32, 100), nn.ReLU(), nn.Linear(100, 32))
cloud = nn.Sequential(nn.Linear(32, 200), nn.ReLU(), nn.Linear(200, 100), nn.ReLU(), nn.Linear(100, 32))

ec_spec = EdgeCloudSpeculativeDecoding(edge, cloud, k=5)

n_steps = 10
total_accepted = 0
total_draft = 0
for _ in range(n_steps):
    x = torch.randn(1, 4, 32)
    accepted, drafted = ec_spec.decode_step(x)
    total_accepted += accepted
    total_draft += drafted

accept_rate = total_accepted / max(total_draft, 1)
speedup = 1 / (1 - 5 * accept_rate) if 5 * accept_rate < 1 else float('inf')

print(f"=== 端云推测解码 ===")
print(f"总draft tokens: {total_draft}")
print(f"总accepted tokens: {total_accepted}")
print(f"接受率: {accept_rate:.2%}")
print(f"理论加速比: {speedup:.2f}x (k=5)")
print(f"\n端云推测解码优势:")
print(f"1. 减少云端计算: 端侧生成候选，云端仅需验证")
print(f"2. 减少通信轮次: 一次上传k个候选，一次验证")
print(f"3. 降低延迟: 端侧draft延迟低，云端验证并行")
print(f"4. 输出等价: 拒绝采样保证与云端大模型分布一致")

print(f"\n--- 不同k值的加速比（假设接受率80%） ---")
assumed_accept = 0.8
for k in [3, 5, 7, 10]:
    sp = 1 / (1 - k * assumed_accept) if k * assumed_accept < 1 else float('inf')
    print(f"  k={k}: 加速比={sp:.2f}x")
print(f"注意: k过大时接受率下降，实际加速比不如理论值")

### 产业级实战：真实模型拆分推理

使用 HuggingFace transformers 加载真实 GPT-2 模型，演示模型拆分推理：前半层在端侧执行，后半层在云端执行，中间特征上传云端而非原始数据。

#### 拆分推理的关键步骤

1. **确定拆分点**：选择第$l$层作为拆分点，前$l$层为端侧模型，后$L-l$层为云端模型
2. **端侧执行**：输入$x$经前$l$层得到中间特征$h_l$
3. **特征传输**：将$h_l$上传到云端（而非原始数据$x$），可量化压缩
4. **云端执行**：云端用后$L-l$层计算最终输出$y$

#### 拆分点选择

$$l^* = \arg\min_l (T_{\text{edge}}(l) + T_{\text{transfer}}(l) + T_{\text{cloud}}(l))$$

中间特征大小：$|h_l| = L_{\text{seq}} \times d_{\text{hidden}} \times 4$ bytes（FP32）

#### 产业界拆分推理实践

- **Splitwise (Microsoft)**：将prefill放在端侧/近端GPU，decode放在远端GPU，优化吞吐
- **Edge-Cloud Collaborative Inference**：根据网络条件动态调整拆分点
- **DistilEdge**：端侧使用蒸馏小模型，云端使用大模型，推测验证加速

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Config, AutoTokenizer
import time

tokenizer = AutoTokenizer.from_pretrained('gpt2')

config_full = GPT2Config.from_pretrained('gpt2')
full_model = GPT2LMHeadModel.from_pretrained('gpt2')
full_model.eval()

n_layers = config_full.n_layer
split_point = n_layers // 2

class EdgeModel(nn.Module):
    """端侧模型: Embedding + 前半层Transformer"""
    def __init__(self, gpt2_model, split_layer):
        super().__init__()
        self.wte = gpt2_model.transformer.wte
        self.wpe = gpt2_model.transformer.wpe
        self.drop = gpt2_model.transformer.drop
        self.h = gpt2_model.transformer.h[:split_layer]

    def forward(self, input_ids):
        position_ids = torch.arange(input_ids.shape[1], device=input_ids.device).unsqueeze(0)
        hidden = self.drop(self.wte(input_ids) + self.wpe(position_ids))
        for block in self.h:
            hidden = block(hidden)[0]
        return hidden

class CloudModel(nn.Module):
    """云端模型: 后半层Transformer + LN + LM Head"""
    def __init__(self, gpt2_model, split_layer):
        super().__init__()
        self.h = gpt2_model.transformer.h[split_layer:]
        self.ln_f = gpt2_model.transformer.ln_f
        self.lm_head = gpt2_model.lm_head

    def forward(self, hidden):
        for block in self.h:
            hidden = block(hidden)[0]
        hidden = self.ln_f(hidden)
        return self.lm_head(hidden)

edge_model = EdgeModel(full_model, split_point)
cloud_model = CloudModel(full_model, split_point)
edge_model.eval()
cloud_model.eval()

input_ids = tokenizer('The future of AI is', return_tensors='pt').input_ids

with torch.no_grad():
    full_output = full_model(input_ids).logits

with torch.no_grad():
    edge_start = time.perf_counter()
    intermediate = edge_model(input_ids)
    edge_time = time.perf_counter() - edge_start

    transfer_bytes = intermediate.numel() * intermediate.element_size()

    cloud_start = time.perf_counter()
    split_output = cloud_model(intermediate)
    cloud_time = time.perf_counter() - cloud_start

max_diff = (full_output - split_output).abs().max().item()

edge_params = sum(p.numel() for p in edge_model.parameters())
cloud_params = sum(p.numel() for p in cloud_model.parameters())
full_params = sum(p.numel() for p in full_model.parameters())

print(f"=== 产业级模型拆分推理: GPT-2 ===")
print(f"总层数: {n_layers}, 拆分点: 第{split_point}层")
print(f"\n{'组件':<15} {'参数量':<15} {'占比':<10} {'延迟(ms)':<12}")
print("-" * 52)
print(f"{'端侧(前半)':<15} {edge_params:<15,} {edge_params/full_params*100:.0f}%{'':<5} {edge_time*1000:<12.2f}")
print(f"{'云端(后半)':<15} {cloud_params:<15,} {cloud_params/full_params*100:.0f}%{'':<5} {cloud_time*1000:<12.2f}")
print(f"{'完整模型':<15} {full_params:<15,} {'100%':<10} {(edge_time+cloud_time)*1000:<12.2f}")

print(f"\n中间特征传输:")
print(f"  特征shape: {intermediate.shape}")
print(f"  传输大小(FP32): {transfer_bytes/1024:.1f} KB")
print(f"  传输大小(FP16): {transfer_bytes/2/1024:.1f} KB")
print(f"  传输大小(INT8): {transfer_bytes/4/1024:.1f} KB")
print(f"  vs 原始数据: {input_ids.numel()*input_ids.element_size()} bytes (token IDs)")
print(f"  数值一致性: 最大差异 = {max_diff:.8f}")

print(f"\n产业界模型拆分实践:")
print(f"1. 隐私: 原始文本不上传，仅上传中间特征")
print(f"2. 量化: 中间特征可INT8量化后传输，减少75%带宽")
print(f"3. 路由: 简单请求端侧独立处理，复杂请求拆分到云端")
print(f"4. 框架: Splitwise (Microsoft) / Edge-Cloud Collaborative Inference")
print(f"5. 安全: TLS 1.3加密中间特征传输，防止中间人攻击")

del full_model, edge_model, cloud_model
gc.collect()

---
### 网络自适应调度

端云协同推理的核心挑战是网络延迟的不确定性。4G/5G/WiFi的延迟和带宽差异巨大，需要根据实时网络条件动态调整推理策略。

#### 网络条件分类

| 网络类型 | 典型延迟(RTT) | 典型带宽 | 适用策略 |
|---------|-------------|---------|----------|
| **WiFi 6** | 5-20ms | 500Mbps+ | 拆分推理(任意拆分点) |
| **5G** | 10-30ms | 100-500Mbps | 拆分推理(偏后拆分点) |
| **4G** | 30-100ms | 10-50Mbps | 端侧独立推理/推测解码 |
| **弱网** | 100ms+ | <10Mbps | 端侧独立推理 |

#### 自适应调度策略

根据实时网络条件选择最优推理路径：

$$\text{Strategy} = \begin{cases} \text{Edge-Only} & \text{if } RTT > T_{\text{threshold}}^{\text{high}} \\ \text{Split}(l^*) & \text{if } T_{\text{threshold}}^{\text{low}} < RTT < T_{\text{threshold}}^{\text{high}} \\ \text{Cloud-Only} & \text{if } RTT < T_{\text{threshold}}^{\text{low}} \end{cases}$$

其中$l^*$根据当前RTT动态计算：

$$l^* = \arg\min_l (T_{\text{edge}}(l) + T_{\text{transfer}}(l, RTT) + T_{\text{cloud}}(l))$$

#### 网络监控指标

- **RTT**：往返延迟，通过心跳包测量
- **可用带宽**：通过传输时间估算
- **丢包率**：影响传输可靠性
- **抖动(jitter)**：延迟波动，影响体验一致性

In [ ]:
class NetworkAdaptiveScheduler:
    """网络自适应调度器"""
    def __init__(self, edge_latency_ms=50, cloud_latency_ms=20, transfer_per_mb_ms=5):
        self.edge_latency_ms = edge_latency_ms
        self.cloud_latency_ms = cloud_latency_ms
        self.transfer_per_mb_ms = transfer_per_mb_ms

    def estimate_split_latency(self, split_layer, total_layers, hidden_dim, seq_len, rtt_ms):
        """估算拆分推理延迟"""
        edge_ratio = (split_layer + 1) / total_layers
        cloud_ratio = 1 - edge_ratio
        edge_time = self.edge_latency_ms * edge_ratio * total_layers
        cloud_time = self.cloud_latency_ms * cloud_ratio * total_layers
        transfer_bytes_mb = seq_len * hidden_dim * 2 / 1e6
        transfer_time = transfer_bytes_mb * self.transfer_per_mb_ms + rtt_ms
        return edge_time + transfer_time + cloud_time

    def select_strategy(self, total_layers, hidden_dim, seq_len, rtt_ms, bandwidth_mbps=100):
        """根据网络条件选择最优策略"""
        edge_only_time = self.edge_latency_ms * total_layers
        cloud_only_time = rtt_ms + self.cloud_latency_ms * total_layers + seq_len * 4 / 1e6 / bandwidth_mbps * 8000

        best_split = None
        best_split_time = float('inf')
        for l in range(1, total_layers):
            t = self.estimate_split_latency(l, total_layers, hidden_dim, seq_len, rtt_ms)
            if t < best_split_time:
                best_split_time = t
                best_split = l

        strategies = {
            'edge_only': edge_only_time,
            'cloud_only': cloud_only_time,
            f'split(layer={best_split})': best_split_time,
        }
        best = min(strategies, key=strategies.get)
        return best, strategies

scheduler = NetworkAdaptiveScheduler()
print("=== 网络自适应调度分析 (7B模型, 32层, hidden=4096) ===")
print(f"\n{'网络条件':<15} {'RTT(ms)':<10} {'带宽(Mbps)':<12} {'最优策略':<20} {'延迟(ms)'}")
print("-" * 70)

network_conditions = [
    ('WiFi 6', 10, 500),
    ('5G', 20, 200),
    ('4G', 50, 30),
    ('弱4G', 100, 10),
    ('极弱网', 200, 5),
]
for name, rtt, bw in network_conditions:
    best, strategies = scheduler.select_strategy(32, 4096, 128, rtt, bw)
    print(f"{name:<15} {rtt:<10} {bw:<12} {best:<20} {strategies[best]:.0f}")

print(f"\n=== 各策略延迟对比 (5G, RTT=20ms) ===")
best, strategies = scheduler.select_strategy(32, 4096, 128, 20, 200)
for name, latency in strategies.items():
    marker = ' ← 最优' if name == best else ''
    print(f"  {name}: {latency:.0f}ms{marker}")

print(f"\n=== 产业实践要点 ===")
print(f"1. 实时网络探测: 每5-10秒测量RTT和带宽，动态调整策略")
print(f"2. 策略切换平滑: 避免在推理过程中途切换，等当前请求完成")
print(f"3. 降级策略: 网络异常时自动降级到端侧推理")
print(f"4. 预测性调度: 根据历史网络数据预测未来网络状态")
print(f"5. 用户体验优先: 延迟阈值(如500ms)比吞吐更重要")

---
### 边缘缓存与KV Cache复用

在端云协同场景中，边缘缓存可以显著减少重复计算和网络传输。

#### 缓存层次

| 缓存类型 | 缓存内容 | 命中率 | 生命周期 |
|---------|---------|--------|----------|
| **Prompt Cache** | 系统提示词的KV Cache | 高(>80%) | 会话级 |
| **Prefix Cache** | 共享前缀的KV Cache | 中(30-60%) | 请求级 |
| **结果Cache** | 完整推理结果 | 低(5-15%) | 时间窗口级 |
| **特征Cache** | 中间层特征(拆分推理) | 中 | 会话级 |

#### Prompt Cache优化

系统提示词（如"你是一个有用的AI助手..."）在每次请求中都重复，其KV Cache可以预计算并缓存：

$$\text{节省计算量} = L_{\text{prefix}} \times N_{\text{layers}} \times 2 \times d_{\text{hidden}} \times d_{\text{kv}}$$

对于7B模型，500 token的系统提示词可节省约$500 \times 32 \times 2 \times 4096 \times 128 = 16.8G$ FLOPs。

#### KV Cache传输优化

在拆分推理中，端侧的KV Cache可以缓存，避免重复计算：
1. **端侧缓存KV**：前$l$层的KV Cache缓存在端侧
2. **增量传输**：仅传输新增token的KV，而非全量
3. **量化压缩**：KV Cache用INT8/INT4量化后传输

In [ ]:
class EdgeCacheManager:
    """边缘缓存管理器"""
    def __init__(self, max_cache_mb=512):
        self.max_cache_mb = max_cache_mb
        self.cache = {}

    def compute_kv_cache_size(self, seq_len, n_layers, n_kv_heads, head_dim, dtype_bytes=2):
        """计算KV Cache大小"""
        return seq_len * n_layers * 2 * n_kv_heads * head_dim * dtype_bytes / 1e6

    def estimate_prompt_cache_savings(self, prefix_len, n_layers, n_kv_heads, head_dim, n_requests):
        """估算Prompt Cache节省"""
        kv_size_mb = self.compute_kv_cache_size(prefix_len, n_layers, n_kv_heads, head_dim)
        flops_per_prefix = prefix_len * n_layers * 2 * (n_kv_heads * head_dim) * (4 * n_kv_heads * head_dim)
        total_flops_saved = flops_per_prefix * n_requests
        return {
            'cache_size_mb': kv_size_mb,
            'flops_saved_per_request': flops_per_prefix,
            'total_flops_saved_gflops': total_flops_saved / 1e9 * n_requests,
            'fits_cache': kv_size_mb <= self.max_cache_mb,
        }

cache_mgr = EdgeCacheManager(max_cache_mb=512)
print("=== 边缘KV Cache分析 (7B模型) ===")
print(f"\n--- KV Cache大小估算 ---")
for seq_len in [128, 512, 2048, 8192]:
    size = cache_mgr.compute_kv_cache_size(seq_len, 32, 32, 128)
    size_int8 = cache_mgr.compute_kv_cache_size(seq_len, 32, 32, 128, dtype_bytes=1)
    print(f"  seq_len={seq_len:<6} FP16: {size:.1f}MB  INT8: {size_int8:.1f}MB")

print(f"\n--- Prompt Cache收益分析 (500 token系统提示词) ---")
for n_requests in [1, 10, 100, 1000]:
    result = cache_mgr.estimate_prompt_cache_savings(500, 32, 32, 128, n_requests)
    print(f"  {n_requests}次请求: 缓存{result['cache_size_mb']:.1f}MB, "
          f"节省{result['flops_saved_per_request']/1e9:.1f}GFLOPs/请求, "
          f"适合缓存: {'是' if result['fits_cache'] else '否'}")

print(f"\n=== 产业实践要点 ===")
print(f"1. Prompt Cache是最高ROI的优化: 系统提示词命中率>80%")
print(f"2. KV Cache量化(INT8)可减少50%缓存大小，精度损失<0.1%")
print(f"3. PagedAttention管理KV Cache内存，消除碎片")
print(f"4. 增量传输: 多轮对话仅传输新增token的KV")
print(f"5. 缓存淘汰策略: LRU适合通用场景，前缀匹配需要定制策略")

---
### 端云模型一致性验证

在端云协同推理中，端侧模型和云端模型可能版本不同、精度不同，需要验证推理结果的一致性。

#### 一致性挑战

| 问题 | 原因 | 影响 |
|------|------|------|
| **精度差异** | 端侧INT4 vs 云端FP16 | 输出分布偏移 |
| **版本差异** | 端侧模型更新滞后 | 功能不一致 |
| **随机性** | Temperature/Top-P采样 | 相同输入不同输出 |
| **数值误差累积** | 浮点运算顺序不同 | 长序列偏差增大 |

#### 一致性验证方法

1. **确定性输出对比**：设置temperature=0，对比端云输出的token级匹配率
2. **分布对比**：对比端云输出的logits分布，计算KL散度
3. **功能测试集**：构建覆盖典型场景的测试集，验证端云输出语义一致
4. **A/B测试**：线上流量对比端云推理的用户满意度

$$D_{\text{KL}}(P_{\text{edge}} \| P_{\text{cloud}}) = \sum_x P_{\text{edge}}(x) \log \frac{P_{\text{edge}}(x)}{P_{\text{cloud}}(x)}$$

产业标准：KL散度 < 0.1 视为可接受的一致性。

In [ ]:
import numpy as np

class ConsistencyVerifier:
    """端云模型一致性验证器"""
    @staticmethod
    def token_match_rate(edge_tokens, cloud_tokens):
        """计算token匹配率"""
        matches = sum(1 for e, c in zip(edge_tokens, cloud_tokens) if e == c)
        return matches / max(len(edge_tokens), 1)

    @staticmethod
    def kl_divergence(p, q, eps=1e-10):
        """计算KL散度"""
        p = np.array(p) + eps
        q = np.array(q) + eps
        p = p / p.sum()
        q = q / q.sum()
        return np.sum(p * np.log(p / q))

    @staticmethod
    def cosine_similarity(a, b):
        """计算余弦相似度"""
        a, b = np.array(a), np.array(b)
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)

verifier = ConsistencyVerifier()

np.random.seed(42)
vocab_size = 100
cloud_logits = np.random.randn(vocab_size)
cloud_probs = np.exp(cloud_logits) / np.exp(cloud_logits).sum()

print("=== 端云一致性验证分析 ===")
print(f"\n--- 不同量化精度的一致性 ---")
print(f"{'量化方案':<20} {'Token匹配率':<15} {'KL散度':<12} {'余弦相似度':<12} {'判定'}")
print("-" * 70)

for quant_name, noise_scale in [('FP16(基准)', 0), ('INT8', 0.01), ('INT4', 0.05), ('INT4(激进)', 0.15)]:
    edge_logits = cloud_logits + np.random.randn(vocab_size) * noise_scale
    edge_probs = np.exp(edge_logits) / np.exp(edge_logits).sum()
    kl = verifier.kl_divergence(edge_probs, cloud_probs)
    cos_sim = verifier.cosine_similarity(edge_logits, cloud_logits)
    edge_tokens = np.argmax(edge_logits[:20]).reshape(1)
    cloud_tokens = np.argmax(cloud_logits[:20]).reshape(1)
    match_rate = 1.0 if edge_tokens[0] == cloud_tokens[0] else 0.0
    verdict = '✓ 通过' if kl < 0.1 else '△ 警告' if kl < 0.5 else '✗ 不通过'
    print(f"{quant_name:<20} {match_rate:<15.1%} {kl:<12.4f} {cos_sim:<12.4f} {verdict}")

print(f"\n--- 多轮对话一致性衰减 ---")
for n_turns in [1, 5, 10, 20, 50]:
    cumulative_error = 0.05 * np.sqrt(n_turns)
    kl_est = cumulative_error ** 2
    verdict = '✓ 通过' if kl_est < 0.1 else '△ 警告' if kl_est < 0.5 else '✗ 不通过'
    print(f"  {n_turns}轮对话: 累积误差≈{cumulative_error:.3f}, KL≈{kl_est:.4f} {verdict}")

print(f"\n=== 产业实践要点 ===")
print(f"1. 每次模型更新必须通过一致性测试: KL<0.1, token匹配率>90%")
print(f"2. 多轮对话的误差累积是主要风险: 长对话需要定期重同步")
print(f"3. 确定性验证(temperature=0)是基础，分布验证是补充")
print(f"4. 端侧模型版本管理: 确保端云模型版本对应关系可追踪")
print(f"5. 灰度验证: 新模型先1%流量验证一致性，通过后逐步放量")

print(f"\n=== 端云协同推理总结 ===")
print(f"核心原则: 端侧优先、云端兜底、智能路由、渐进降级")
print(f"关键指标: 端侧命中率>70%、端云切换延迟<100ms、一致性KL<0.1")
print(f"通信优化: Protobuf序列化+流式传输+压缩，减少端云数据量")
print(f"运维保障: 灰度发布+一致性验证+监控告警+快速回滚")

---
## 总结与最佳实践

### 端云协同推理的核心原则

1. **端侧优先**：尽可能在端侧完成推理，减少云端依赖和延迟
2. **智能路由**：根据任务复杂度、网络条件、隐私需求动态选择推理路径
3. **一致性保障**：端云模型输出必须一致，KL散度<0.1，token匹配率>90%
4. **渐进降级**：端侧→云端→缓存回复，确保服务可用性

### 端云通信协议选型

| 协议 | 适用场景 | 延迟 | 吞吐 | 流式支持 |
|------|---------|------|------|----------|
| **gRPC** | 端云推理调用 | 低(~10ms) | 高 | ✓ (streaming RPC) |
| **HTTP/2+SSE** | Web端流式输出 | 中(~50ms) | 中 | ✓ (Server-Sent Events) |
| **WebSocket** | 双向实时通信 | 低 | 高 | ✓ (全双工) |
| **MQTT** | IoT设备 | 中 | 低 | 有限 |

### 产业级端云协同Checklist

- [ ] 确定端侧/云端的模型拆分策略
- [ ] 实现自适应推理路由（基于复杂度+网络+隐私）
- [ ] 建立端云模型一致性验证流程
- [ ] 选择通信协议并实现流式传输
- [ ] 实现边缘缓存和KV Cache复用
- [ ] 建立降级策略（端侧→云端→缓存回复）
- [ ] 部署监控告警（延迟P99、错误率、缓存命中率）
- [ ] 建立灰度发布和快速回滚机制